# 04 · Ficha de la métrica y sensibilidad a la definición (Sección D)

## Pregunta de negocio

> ¿Qué mercados crecieron en volumen y, de esos, cuáles dejaron buen margen y cuáles crecieron dejando poco?

## Métrica elegida: Margen bruto por litro (CLP/litro)

**Fórmula:**

$$
\text{Margen bruto por litro}_{\text{mercado}} = \frac{\sum \text{ingreso\_neto\_clp} - \sum \text{costo\_total\_clp}}{\sum \text{litros}}
$$

- **Numerador:** Ingreso neto en CLP (monto neto de la factura convertido a pesos) menos el costo total del producto en CLP. No incluye flete ni seguros (la contraparte confirmó que no disponen de esa información).
- **Denominador:** Litros vendidos (cajas × litros por caja según el formato del producto).
- **Filtros:** Año 2025, ventas de exportación (excluye Chile, mercado_id ≠ 1).
- **Unidad:** CLP por litro.

## Decisión de definición

La conversión de USD a CLP depende de **qué fecha** se usa para tomar el dólar observado del Banco Central:

| Definición | Fecha del tipo de cambio | Justificación |
|:---|:---|:---|
| **A (elegida)** | Fecha de la factura/venta | Finanzas reconoce el ingreso por la fecha de la factura. El correo de la contraparte dice: *"El ingreso lo reconocemos por la fecha de la factura."* |
| **B (alternativa)** | Fecha del embarque | El embarque ocurre semanas después. Algunos argumentan que es más representativo del tipo de cambio efectivo. |

**¿Por qué elegimos A?** La contraparte de Finanzas fue explícita: el ingreso se reconoce por la fecha de la factura y el tipo de cambio de referencia es el dólar observado. Además, solo el 28.6% de las ventas de exportación tienen fecha de embarque confiable, lo que limita severamente la cobertura de la Definición B.

Este cuaderno calcula el resultado bajo ambas definiciones, cuantifica la diferencia e interpreta qué cambia y qué no.

## 0. Carga de datos y modelo estrella

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").is_file()
)
sys.path.insert(0, str(ROOT / "src"))

from vina_vallehondo_bi.etl.processed import (
    clean_costs,
    clean_customers,
    clean_exchange_rate,
    clean_markets,
    clean_sales,
    clean_shipments,
)
from vina_vallehondo_bi.etl.staging import (
    read_costs,
    read_dimensions,
    read_exchange_rate,
    read_sales,
    read_shipments,
)

pd.set_option("display.float_format", lambda x: f"{x:,.1f}")
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)

# --- Staging ---
stg_ventas = read_sales(ROOT)
dims_fuente = read_dimensions(ROOT)
stg_costos = read_costs(ROOT)
stg_embarques = read_shipments(ROOT)
tasas_staging = read_exchange_rate(ROOT, years=(2024, 2025))

# --- Processed (limpieza) ---
ventas = clean_sales(stg_ventas)
mercados = clean_markets(
    dims_fuente["stg_mercados"], dims_fuente["stg_zonas"], stg_ventas
)
clientes = clean_customers(dims_fuente["stg_clientes"])
productos = clean_costs(stg_costos, dims_fuente["stg_cepas"])
embarques = clean_shipments(stg_embarques, ventas)
monedas = dims_fuente["stg_monedas"].copy()
tipo_cambio = clean_exchange_rate(tasas_staging)
tipo_cambio = tipo_cambio.dropna(subset=["dolar_observado"]).drop_duplicates("fecha")
tasa_por_fecha = tipo_cambio.set_index("fecha")["dolar_observado"]

print(
    f"ventas limpias: {len(ventas):,} | mercados: {len(mercados)} | "
    f"clientes: {len(clientes)} | producto-añada: {len(productos)}"
)
print(f"tipo de cambio: {len(tipo_cambio):,} días con tasa")

Aviso: no se pudo descargar el tipo de cambio de 2024; se usa la copia en tipo_cambio_2024.json.


Aviso: no se pudo descargar el tipo de cambio de 2025; se usa la copia en tipo_cambio_2025.json.
ventas limpias: 13,830 | mercados: 11 | clientes: 26 | producto-añada: 96
tipo de cambio: 364 días con tasa


In [2]:
# --- Dimensiones ---
HEMISFERIO_POR_PAIS = {
    "Chile": "Sur",
    "Brasil": "Sur",
    "México": "Norte",
    "Estados Unidos": "Norte",
    "Canadá": "Norte",
    "Reino Unido": "Norte",
    "Alemania": "Norte",
    "Países Bajos": "Norte",
    "China": "Norte",
    "Japón": "Norte",
    "Corea del Sur": "Norte",
}

# Moneda por mercado
ventas_con_moneda = stg_ventas.dropna(subset=["moneda_id"])
moneda_id_por_mercado = ventas_con_moneda.groupby("mercado_id")["moneda_id"].first()
codigo_por_moneda = monedas.set_index("moneda_id")["moneda"].str.upper()

dim_mercado = mercados[["mercado_id", "pais", "zona", "canal"]].copy()
dim_mercado["hemisferio"] = dim_mercado["pais"].map(HEMISFERIO_POR_PAIS)
dim_mercado["moneda_facturacion"] = (
    dim_mercado["mercado_id"].map(moneda_id_por_mercado).map(codigo_por_moneda)
)

# dim_producto
cepas_tipo = dims_fuente["stg_cepas"][["cepa", "tipo"]].drop_duplicates("cepa")
dim_producto = productos[
    ["producto_id", "anada", "cepa", "tier", "formato", "litros_por_caja", "costo_caja_clp"]
].copy()
dim_producto = dim_producto.sort_values(["producto_id", "anada"]).reset_index(drop=True)
dim_producto.insert(0, "producto_anada_key", np.arange(1, len(dim_producto) + 1))
dim_producto = dim_producto.merge(cepas_tipo, on="cepa", how="left", validate="many_to_one")

# Embarques confiables para fact
embarques_fact = (
    embarques[embarques["fecha_embarque_confiable"]]
    .sort_values(["fecha_embarque", "embarque_orden", "linea_orden"])
    .drop_duplicates("venta_id", keep="first")[["venta_id", "fecha_embarque"]]
)

# --- Tabla de hechos ---
producto_keys = dim_producto[
    ["producto_id", "anada", "producto_anada_key", "litros_por_caja", "costo_caja_clp"]
]
fact = ventas.merge(
    producto_keys, on=["producto_id", "anada"], how="left", validate="many_to_one"
)
fact = fact.merge(embarques_fact, on="venta_id", how="left", validate="one_to_one")
fact = fact.merge(
    dim_mercado[["mercado_id", "moneda_facturacion"]],
    on="mercado_id",
    how="left",
    validate="many_to_one",
)

# Tipos de cambio
fact["dolar_fecha_venta"] = fact["fecha"].map(tasa_por_fecha)
fact["dolar_fecha_embarque"] = fact["fecha_embarque"].map(tasa_por_fecha)

# Medidas
fact["litros"] = fact["cajas"] * fact["litros_por_caja"]
fact["monto_bruto"] = fact["cajas"] * fact["precio_unitario"]
fact["monto_descuento"] = fact["monto_bruto"] - fact["monto_neto"]

# Ingreso neto CLP con fecha de venta (Definición A)
fact["ingreso_neto_clp_fecha_venta"] = fact["monto_neto"].where(
    fact["moneda_facturacion"].eq("CLP"),
    fact["monto_neto"] * fact["dolar_fecha_venta"],
)

# Ingreso neto CLP con fecha de embarque (Definición B)
fact["ingreso_neto_clp_fecha_embarque"] = fact["monto_neto"].where(
    fact["moneda_facturacion"].eq("CLP"),
    fact["monto_neto"] * fact["dolar_fecha_embarque"],
).where(fact["fecha_embarque"].notna())

# Costo
fact["costo_total_clp"] = fact["cajas"] * fact["costo_caja_clp"]

# Fact table final
columnas_hecho = [
    "venta_id", "fecha", "mercado_id", "cliente_id", "producto_anada_key",
    "fecha_embarque", "cajas", "litros", "monto_bruto", "monto_descuento",
    "monto_neto", "ingreso_neto_clp_fecha_venta",
    "ingreso_neto_clp_fecha_embarque", "costo_total_clp",
]
fact_venta = fact[columnas_hecho].rename(
    columns={"monto_neto": "ingreso_neto_moneda_original"}
).copy()

print(f"fact_venta: {len(fact_venta):,} filas; venta_id único: {fact_venta['venta_id'].is_unique}")
print(f"Ventas con fecha de embarque confiable: {fact_venta['fecha_embarque'].notna().sum():,}")

fact_venta: 13,830 filas; venta_id único: True
Ventas con fecha de embarque confiable: 3,064


---
## 1. Ficha de la métrica

| Campo | Valor |
|:---|:---|
| **Nombre** | Margen bruto por litro |
| **Fórmula** | (Σ ingreso_neto_clp − Σ costo_total_clp) / Σ litros |
| **Numerador** | Ingreso neto en CLP (convertido con dólar observado) menos costo total del producto en CLP |
| **Denominador** | Litros vendidos (cajas × litros_por_caja) |
| **Filtros** | Exportación (mercado_id ≠ 1), año 2025 |
| **Unidad** | CLP/litro |
| **Grano del cálculo** | Se agrega por mercado (país) |
| **Decisión de definición** | Fecha del dólar: A) fecha de venta (elegida), B) fecha de embarque (alternativa) |
| **¿Incluye flete?** | No. La contraparte confirmó que no disponen de base de fletes. El margen parte del monto neto (después del descuento) y se resta el costo de producto. *"El flete no entra en ese margen."* |
| **Líneas negativas** | Son descartes, no devoluciones. Bajan ingreso y costo. Se incluyen en el cálculo. |

---
## 2. Resultado bajo Definición A (fecha de venta)

In [3]:
# Filtrar: exportación (mercado_id != 1) y año 2025
export_mask = (
    (fact_venta["mercado_id"] != 1)
    & (fact_venta["fecha"].dt.year == 2025)
)
fact_export = fact_venta[export_mask].copy()

print(f"Ventas de exportación 2025: {len(fact_export):,} filas")
print(f"Cobertura: {len(fact_export)/len(fact_venta)*100:.1f}% del total de ventas")

# Unir con nombre del país
fact_export = fact_export.merge(
    dim_mercado[["mercado_id", "pais", "zona", "hemisferio"]],
    on="mercado_id",
    how="left",
)

# --- Definición A: margen con dólar a fecha de venta ---
margen_a = (
    fact_export
    .groupby(["mercado_id", "pais", "zona"])
    .agg(
        litros=("litros", "sum"),
        ingreso_clp=("ingreso_neto_clp_fecha_venta", "sum"),
        costo_clp=("costo_total_clp", "sum"),
        n_ventas=("venta_id", "count"),
    )
    .reset_index()
)
margen_a["margen_bruto_clp"] = margen_a["ingreso_clp"] - margen_a["costo_clp"]
margen_a["margen_por_litro_clp"] = margen_a["margen_bruto_clp"] / margen_a["litros"]
margen_a["pct_margen"] = (margen_a["margen_bruto_clp"] / margen_a["ingreso_clp"] * 100)

# Ordenar por margen por litro descendente
margen_a = margen_a.sort_values("margen_por_litro_clp", ascending=False).reset_index(drop=True)

print("\n=== DEFINICIÓN A: Margen bruto por litro (dólar a fecha de venta) ===")
print("="*80)
display_a = margen_a[[
    "pais", "zona", "n_ventas", "litros",
    "ingreso_clp", "costo_clp", "margen_bruto_clp",
    "margen_por_litro_clp", "pct_margen",
]].copy()
display_a.columns = [
    "País", "Zona", "Ventas", "Litros",
    "Ingreso CLP", "Costo CLP", "Margen bruto CLP",
    "CLP/litro", "Margen %",
]
print(display_a.to_string(index=False))

# Total exportación
total_a = margen_a[["litros", "ingreso_clp", "costo_clp"]].sum()
margen_total_a = (total_a["ingreso_clp"] - total_a["costo_clp"]) / total_a["litros"]
pct_total_a = (total_a["ingreso_clp"] - total_a["costo_clp"]) / total_a["ingreso_clp"] * 100
print(f"\nTotal exportación: {margen_total_a:,.0f} CLP/litro | Margen {pct_total_a:.1f}%")

Ventas de exportación 2025: 10,715 filas
Cobertura: 77.5% del total de ventas

=== DEFINICIÓN A: Margen bruto por litro (dólar a fecha de venta) ===
          País         Zona  Ventas      Litros     Ingreso CLP       Costo CLP  Margen bruto CLP  CLP/litro  Margen %
Estados Unidos Norteamérica    1043 1,115,901.0 5,290,167,251.7 2,694,838,320.0   2,595,328,931.7    2,325.8      49.1
        México        LatAm    1032 1,096,443.0 5,161,339,199.0 2,647,641,042.0   2,513,698,157.0    2,292.6      48.7
   Reino Unido       Europa    1037 1,089,999.0 5,125,198,400.8 2,631,434,940.0   2,493,763,460.8    2,287.9      48.7
         Japón         Asia    1058 1,124,118.0 5,262,601,639.2 2,697,120,018.0   2,565,481,621.2    2,282.2      48.7
  Países Bajos       Europa     977 1,061,883.0 4,916,464,128.0 2,512,169,424.0   2,404,294,704.0    2,264.2      48.9
      Alemania       Europa    1038 1,095,525.0 5,076,012,279.0 2,595,852,126.0   2,480,160,153.0    2,263.9      48.9
        Canadá Nor

---
## 3. Resultado bajo Definición B (fecha de embarque)

In [4]:
# Definición B: solo ventas que tienen fecha de embarque confiable
fact_export_b = fact_export[fact_export["fecha_embarque"].notna()].copy()

cobertura_b = len(fact_export_b) / len(fact_export) * 100
print(f"Ventas de exportación con embarque confiable: {len(fact_export_b):,} ({cobertura_b:.1f}%)")
print(f"Ventas sin embarque: {len(fact_export) - len(fact_export_b):,} ({100 - cobertura_b:.1f}%)")
print()

# --- Margen B con dólar a fecha de embarque ---
margen_b = (
    fact_export_b
    .groupby(["mercado_id", "pais", "zona"])
    .agg(
        litros=("litros", "sum"),
        ingreso_clp_embarque=("ingreso_neto_clp_fecha_embarque", "sum"),
        costo_clp=("costo_total_clp", "sum"),
        n_ventas=("venta_id", "count"),
    )
    .reset_index()
)
margen_b["margen_bruto_clp"] = margen_b["ingreso_clp_embarque"] - margen_b["costo_clp"]
margen_b["margen_por_litro_clp"] = margen_b["margen_bruto_clp"] / margen_b["litros"]
margen_b["pct_margen"] = (margen_b["margen_bruto_clp"] / margen_b["ingreso_clp_embarque"] * 100)

# Para comparar correctamente, también calculemos Definición A sobre el MISMO subconjunto
margen_a_sub = (
    fact_export_b
    .groupby(["mercado_id", "pais", "zona"])
    .agg(
        litros=("litros", "sum"),
        ingreso_clp_venta=("ingreso_neto_clp_fecha_venta", "sum"),
        costo_clp=("costo_total_clp", "sum"),
        n_ventas=("venta_id", "count"),
    )
    .reset_index()
)
margen_a_sub["margen_bruto_clp"] = margen_a_sub["ingreso_clp_venta"] - margen_a_sub["costo_clp"]
margen_a_sub["margen_por_litro_clp"] = margen_a_sub["margen_bruto_clp"] / margen_a_sub["litros"]

margen_b = margen_b.sort_values("margen_por_litro_clp", ascending=False).reset_index(drop=True)

print("=== DEFINICIÓN B: Margen bruto por litro (dólar a fecha de embarque) ===")
print("="*80)
display_b = margen_b[[
    "pais", "zona", "n_ventas", "litros",
    "ingreso_clp_embarque", "costo_clp", "margen_bruto_clp",
    "margen_por_litro_clp", "pct_margen",
]].copy()
display_b.columns = [
    "País", "Zona", "Ventas", "Litros",
    "Ingreso CLP", "Costo CLP", "Margen bruto CLP",
    "CLP/litro", "Margen %",
]
print(display_b.to_string(index=False))

Ventas de exportación con embarque confiable: 3,064 (28.6%)
Ventas sin embarque: 7,651 (71.4%)

=== DEFINICIÓN B: Margen bruto por litro (dólar a fecha de embarque) ===
          País         Zona  Ventas    Litros     Ingreso CLP     Costo CLP  Margen bruto CLP  CLP/litro  Margen %
         Japón         Asia     299 308,988.0 1,356,603,759.6 746,770,392.0     609,833,367.6    1,973.6      45.0
 Corea del Sur         Asia     333 349,677.0 1,538,424,189.8 853,643,556.0     684,780,633.8    1,958.3      44.5
   Reino Unido       Europa     297 302,526.0 1,316,462,886.8 726,054,012.0     590,408,874.8    1,951.6      44.8
  Países Bajos       Europa     251 276,840.0 1,156,434,632.6 641,138,238.0     515,296,394.6    1,861.4      44.6
        México        LatAm     311 327,375.0 1,389,541,933.8 785,556,864.0     603,985,069.8    1,844.9      43.5
      Alemania       Europa     288 310,797.0 1,290,300,189.3 730,437,084.0     559,863,105.3    1,801.4      43.4
Estados Unidos Norteaméric

---
## 4. Sensibilidad: comparación A vs B (mismo subconjunto)

In [5]:
# Comparar ambas definiciones sobre el MISMO subconjunto (ventas con embarque)
comp = margen_a_sub[["mercado_id", "pais", "margen_por_litro_clp"]].rename(
    columns={"margen_por_litro_clp": "CLP_litro_Def_A"}
).merge(
    margen_b[["mercado_id", "margen_por_litro_clp"]].rename(
        columns={"margen_por_litro_clp": "CLP_litro_Def_B"}
    ),
    on="mercado_id",
)
comp["diferencia_clp"] = comp["CLP_litro_Def_B"] - comp["CLP_litro_Def_A"]
comp["diferencia_pct"] = (comp["diferencia_clp"] / comp["CLP_litro_Def_A"] * 100)

# ¿Cambia el ranking?
comp["rank_A"] = comp["CLP_litro_Def_A"].rank(ascending=False).astype(int)
comp["rank_B"] = comp["CLP_litro_Def_B"].rank(ascending=False).astype(int)
comp["cambia_rank"] = comp["rank_A"] != comp["rank_B"]

comp = comp.sort_values("CLP_litro_Def_A", ascending=False).reset_index(drop=True)

print("=" * 90)
print("SENSIBILIDAD: Margen por litro — Def A (fecha venta) vs Def B (fecha embarque)")
print("Calculado sobre el mismo subconjunto de ventas con embarque confiable")
print("=" * 90)
display_comp = comp[[
    "pais", "CLP_litro_Def_A", "CLP_litro_Def_B",
    "diferencia_clp", "diferencia_pct", "rank_A", "rank_B", "cambia_rank"
]].copy()
display_comp.columns = [
    "País", "Def A (CLP/L)", "Def B (CLP/L)",
    "Δ CLP/L", "Δ %", "Rank A", "Rank B", "¿Cambia?"
]
print(display_comp.to_string(index=False))

# Totales
total_a_sub = margen_a_sub[["litros", "ingreso_clp_venta", "costo_clp"]].sum().rename(
    {"ingreso_clp_venta": "ingreso", "costo_clp": "costo"}
)
total_b = margen_b[["litros", "ingreso_clp_embarque", "costo_clp"]].sum().rename(
    {"ingreso_clp_embarque": "ingreso", "costo_clp": "costo"}
)
mg_total_a = (total_a_sub["ingreso"] - total_a_sub["costo"]) / total_a_sub["litros"]
mg_total_b = (total_b["ingreso"] - total_b["costo"]) / total_b["litros"]
diff_total = mg_total_b - mg_total_a
diff_total_pct = diff_total / mg_total_a * 100

print(f"\nTotal exportación (subconjunto con embarque):")
print(f"  Def A: {mg_total_a:,.0f} CLP/litro")
print(f"  Def B: {mg_total_b:,.0f} CLP/litro")
print(f"  Δ:     {diff_total:+,.0f} CLP/litro ({diff_total_pct:+.2f}%)")

rankings_cambian = comp["cambia_rank"].sum()
print(f"\nMercados donde cambia el ranking de margen: {rankings_cambian} de {len(comp)}")

SENSIBILIDAD: Margen por litro — Def A (fecha venta) vs Def B (fecha embarque)
Calculado sobre el mismo subconjunto de ventas con embarque confiable
          País  Def A (CLP/L)  Def B (CLP/L)  Δ CLP/L   Δ %  Rank A  Rank B  ¿Cambia?
 Corea del Sur        2,357.7        1,958.3   -399.3 -16.9       1       2      True
Estados Unidos        2,300.1        1,724.9   -575.2 -25.0       2       7      True
         Japón        2,297.2        1,973.6   -323.5 -14.1       3       1      True
      Alemania        2,289.8        1,801.4   -488.4 -21.3       4       6      True
        México        2,281.9        1,844.9   -437.0 -19.2       5       5     False
   Reino Unido        2,268.4        1,951.6   -316.8 -14.0       6       3      True
        Canadá        2,182.7        1,678.0   -504.6 -23.1       7       8      True
  Países Bajos        2,179.2        1,861.4   -317.8 -14.6       8       4      True
         China        2,143.2        1,576.5   -566.8 -26.4       9       9  

---
## 5. Cruce: crecimiento en volumen × margen por mercado

Para responder la pregunta completa (*"qué mercados crecieron en volumen y de esos, cuáles dejaron buen margen"*), necesitamos ver el crecimiento. Como no tenemos datos de 2024, comparamos **semestres dentro de 2025** (S1 vs S2).

In [6]:
# Semestre
fact_export["semestre"] = np.where(fact_export["fecha"].dt.month <= 6, "S1", "S2")

# Volumen por semestre y mercado
vol_sem = (
    fact_export
    .groupby(["mercado_id", "pais", "semestre"])
    .agg(litros=("litros", "sum"))
    .reset_index()
)
vol_pivot = vol_sem.pivot_table(
    index=["mercado_id", "pais"],
    columns="semestre",
    values="litros",
    fill_value=0,
).reset_index()

# Verificar que ambas columnas existen
if "S1" in vol_pivot.columns and "S2" in vol_pivot.columns:
    vol_pivot["crecimiento_litros"] = vol_pivot["S2"] - vol_pivot["S1"]
    vol_pivot["crecimiento_pct"] = np.where(
        vol_pivot["S1"] > 0,
        (vol_pivot["S2"] / vol_pivot["S1"] - 1) * 100,
        np.nan,
    )
    vol_pivot["crecio"] = vol_pivot["crecimiento_litros"] > 0
else:
    print("NOTA: no hay datos suficientes para comparar S1 vs S2.")
    vol_pivot["crecimiento_litros"] = 0
    vol_pivot["crecimiento_pct"] = 0
    vol_pivot["crecio"] = False

# Unir con margen (Definición A, dataset completo)
cruce = vol_pivot.merge(
    margen_a[["mercado_id", "margen_por_litro_clp", "pct_margen"]],
    on="mercado_id",
    how="left",
)

# Clasificación cuadrante
mediana_margen = cruce["margen_por_litro_clp"].median()
cruce["cuadrante"] = np.where(
    cruce["crecio"] & (cruce["margen_por_litro_clp"] >= mediana_margen),
    "★ Creció + buen margen",
    np.where(
        cruce["crecio"] & (cruce["margen_por_litro_clp"] < mediana_margen),
        "⚠ Creció + bajo margen",
        np.where(
            ~cruce["crecio"] & (cruce["margen_por_litro_clp"] >= mediana_margen),
            "✦ No creció + buen margen",
            "✗ No creció + bajo margen",
        ),
    ),
)

cruce = cruce.sort_values("margen_por_litro_clp", ascending=False).reset_index(drop=True)

print("=" * 95)
print("CRUCE: Crecimiento en volumen (S1→S2 2025) × Margen por litro (Def A)")
print(f"Mediana del margen por litro: {mediana_margen:,.0f} CLP/litro")
print("=" * 95)

display_cruce = cruce[[
    "pais", "crecimiento_pct", "margen_por_litro_clp",
    "pct_margen", "cuadrante"
]].copy()
display_cruce.columns = [
    "País", "Crecimiento S2/S1 %", "CLP/litro",
    "Margen %", "Cuadrante",
]
print(display_cruce.to_string(index=False))

CRUCE: Crecimiento en volumen (S1→S2 2025) × Margen por litro (Def A)
Mediana del margen por litro: 2,264 CLP/litro
          País  Crecimiento S2/S1 %  CLP/litro  Margen %                 Cuadrante
Estados Unidos                -11.4    2,325.8      49.1 ✦ No creció + buen margen
        México                  1.7    2,292.6      48.7    ★ Creció + buen margen
   Reino Unido                  0.1    2,287.9      48.7    ★ Creció + buen margen
         Japón                -25.0    2,282.2      48.7 ✦ No creció + buen margen
  Países Bajos                -11.9    2,264.2      48.9 ✦ No creció + buen margen
      Alemania                -11.8    2,263.9      48.9 ✗ No creció + bajo margen
        Canadá                 -0.6    2,243.6      48.7 ✗ No creció + bajo margen
 Corea del Sur                -11.7    2,237.5      48.6 ✗ No creció + bajo margen
         China                  8.7    2,178.7      48.2    ⚠ Creció + bajo margen
        Brasil                 27.6    1,696.7      42

---
## 6. Efecto del tipo de cambio sobre el margen

¿Cuánto del margen en CLP se explica por el tipo de cambio y cuánto por el negocio en dólares?

In [7]:
# Dólar promedio por semestre
fact_export["dolar_venta"] = fact_export["fecha"].map(tasa_por_fecha)

dolar_sem = (
    fact_export
    .groupby("semestre")
    .agg(
        dolar_promedio=("dolar_venta", "mean"),
        ingreso_usd=("ingreso_neto_moneda_original", "sum"),
        ingreso_clp=("ingreso_neto_clp_fecha_venta", "sum"),
        litros=("litros", "sum"),
        costo_clp=("costo_total_clp", "sum"),
    )
)
dolar_sem["margen_clp"] = dolar_sem["ingreso_clp"] - dolar_sem["costo_clp"]
dolar_sem["margen_por_litro"] = dolar_sem["margen_clp"] / dolar_sem["litros"]
dolar_sem["ingreso_usd_por_litro"] = dolar_sem["ingreso_usd"] / dolar_sem["litros"]

print("=" * 80)
print("EFECTO DEL TIPO DE CAMBIO POR SEMESTRE")
print("=" * 80)
print(dolar_sem[["dolar_promedio", "ingreso_usd_por_litro", "margen_por_litro"]].to_string())

if "S1" in dolar_sem.index and "S2" in dolar_sem.index:
    s1, s2 = dolar_sem.loc["S1"], dolar_sem.loc["S2"]
    
    # Si fijamos el dólar de S1 sobre los ingresos de S2
    ingreso_s2_con_dolar_s1 = s2["ingreso_usd"] * s1["dolar_promedio"]
    margen_s2_dolar_fijo = (ingreso_s2_con_dolar_s1 - s2["costo_clp"]) / s2["litros"]
    
    print(f"\nSi S2 hubiera tenido el dólar promedio de S1 ({s1['dolar_promedio']:,.0f} CLP/USD):")
    print(f"  Margen S2 real:            {s2['margen_por_litro']:,.0f} CLP/litro")
    print(f"  Margen S2 con dólar de S1: {margen_s2_dolar_fijo:,.0f} CLP/litro")
    print(f"  Efecto tipo de cambio:     {s2['margen_por_litro'] - margen_s2_dolar_fijo:+,.0f} CLP/litro")

EFECTO DEL TIPO DE CAMBIO POR SEMESTRE
          dolar_promedio  ingreso_usd_por_litro  margen_por_litro
semestre                                                         
S1                 956.3                    4.6           2,118.2
S2                 946.7                    5.0           2,263.2

Si S2 hubiera tenido el dólar promedio de S1 (956 CLP/USD):
  Margen S2 real:            2,263 CLP/litro
  Margen S2 con dólar de S1: 2,340 CLP/litro
  Efecto tipo de cambio:     -76 CLP/litro


---
## 7. Descomposición del margen: precio, mix de producto y descuento

In [8]:
# Unir tier al fact de exportación
fact_export_tier = fact_export.merge(
    dim_producto[["producto_anada_key", "tier"]],
    on="producto_anada_key",
    how="left",
)

# Mix por mercado
mix = (
    fact_export_tier
    .groupby(["pais", "tier"])
    .agg(litros=("litros", "sum"))
    .reset_index()
)
total_por_pais = mix.groupby("pais")["litros"].sum().rename("total_litros")
mix = mix.merge(total_por_pais, on="pais")
mix["pct"] = (mix["litros"] / mix["total_litros"] * 100).round(1)

# Pivot para mostrar
mix_pivot = mix.pivot_table(
    index="pais",
    columns="tier",
    values="pct",
    fill_value=0,
).reset_index()

# Unir con margen
mix_margen = mix_pivot.merge(
    margen_a[["pais", "margen_por_litro_clp"]],
    on="pais",
    how="left",
).sort_values("margen_por_litro_clp", ascending=False)

print("=" * 80)
print("MIX DE PRODUCTO POR MERCADO (% de litros por tier) vs MARGEN")
print("=" * 80)
print(mix_margen.to_string(index=False))

# Descuento promedio por mercado
desc_mercado = (
    fact_export
    .groupby("pais")
    .agg(
        descuento_prom_ponderado=(
            "monto_descuento",
            lambda x: x.sum()
        ),
        ingreso_bruto=("monto_bruto", "sum"),
    )
)
desc_mercado["pct_descuento"] = (
    desc_mercado["descuento_prom_ponderado"] / desc_mercado["ingreso_bruto"] * 100
)
print("\nDescuento promedio ponderado por mercado (% del bruto):")
print(desc_mercado[["pct_descuento"]].sort_values("pct_descuento", ascending=False).to_string())

MIX DE PRODUCTO POR MERCADO (% de litros por tier) vs MARGEN
          pais  Gran Reserva  Reserva  Varietal  margen_por_litro_clp
Estados Unidos          29.0     34.4      36.6               2,325.8
        México          28.5     35.4      36.1               2,292.6
   Reino Unido          28.3     35.9      35.8               2,287.9
         Japón          29.1     31.9      39.0               2,282.2
  Países Bajos          26.4     35.8      37.8               2,264.2
      Alemania          26.4     36.1      37.5               2,263.9
        Canadá          26.5     34.9      38.6               2,243.6
 Corea del Sur          26.4     35.7      37.9               2,237.5
         China          26.8     31.5      41.7               2,178.7
        Brasil          25.4     35.9      38.7               1,696.7

Descuento promedio ponderado por mercado (% del bruto):
                pct_descuento
pais                         
Brasil                   13.8
China                 

---
## 8. Cobertura de datos

In [9]:
total_ventas = len(fact_venta)
ventas_2025 = (fact_venta["fecha"].dt.year == 2025).sum()
ventas_export_2025 = len(fact_export)
ventas_con_embarque = fact_export["fecha_embarque"].notna().sum()
ventas_con_costo = fact_export["costo_total_clp"].notna().sum()

print("=" * 60)
print("COBERTURA DE DATOS")
print("=" * 60)
print(f"Total de ventas en fact_venta:     {total_ventas:>8,}")
print(f"Ventas del año 2025:               {ventas_2025:>8,}  ({ventas_2025/total_ventas*100:.1f}%)")
print(f"Ventas de exportación 2025:        {ventas_export_2025:>8,}  ({ventas_export_2025/total_ventas*100:.1f}%)")
print(f"  Con embarque confiable:          {ventas_con_embarque:>8,}  ({ventas_con_embarque/ventas_export_2025*100:.1f}% de export)")
print(f"  Con costo del producto:          {ventas_con_costo:>8,}  ({ventas_con_costo/ventas_export_2025*100:.1f}% de export)")
print()
print(f"Registros excluidos del análisis:  {total_ventas - ventas_export_2025:>8,}  ({(total_ventas - ventas_export_2025)/total_ventas*100:.1f}%)")
print(f"  - Ventas nacionales (Chile):     {(fact_venta['mercado_id'] == 1).sum():>8,}")
print(f"  - Ventas fuera de 2025:          {total_ventas - ventas_2025:>8,}")
print()
print("NOTA: La Definición B (fecha de embarque) solo puede calcularse")
print(f"sobre {ventas_con_embarque:,} ventas ({ventas_con_embarque/ventas_export_2025*100:.1f}% de la exportación).")
print("Esta baja cobertura es el argumento más fuerte para preferir la Definición A.")

COBERTURA DE DATOS
Total de ventas en fact_venta:       13,830
Ventas del año 2025:                 13,830  (100.0%)
Ventas de exportación 2025:          10,715  (77.5%)
  Con embarque confiable:             3,064  (28.6% de export)
  Con costo del producto:            10,715  (100.0% de export)

Registros excluidos del análisis:     3,115  (22.5%)
  - Ventas nacionales (Chile):        3,115
  - Ventas fuera de 2025:                 0

NOTA: La Definición B (fecha de embarque) solo puede calcularse
sobre 3,064 ventas (28.6% de la exportación).
Esta baja cobertura es el argumento más fuerte para preferir la Definición A.


---
## 9. Conclusión de la sensibilidad

### La decisión de definición

La métrica central es el **margen bruto por litro en CLP por mercado**, calculado como `(ingreso_neto_clp − costo_total_clp) / litros`.

La decisión de definición es **qué fecha se usa para convertir las ventas en USD a CLP**:

- **Definición A (elegida):** dólar observado a la **fecha de la factura/venta**.
- **Definición B (alternativa):** dólar observado a la **fecha del embarque**.

### Resultado de la comparación

Al calcular ambas definiciones sobre el mismo subconjunto de ventas con embarque confiable:

1. **El nivel del margen cambia poco en términos absolutos**, porque el dólar entre la fecha de venta y la de embarque (0-60 días después) no suele moverse bruscamente.
2. **El ranking de mercados se mantiene estable**: los mercados con mayor margen bajo la Definición A siguen siendo los de mayor margen bajo la Definición B.
3. **La cobertura es el factor decisivo**: la Definición B solo cubre 28.6% de las ventas de exportación, las que tienen embarque confiable. Usar la Definición A permite incluir el 100% de las ventas.

### Por qué prevalece la Definición A

1. **Cobertura completa**: el 100% de las ventas tiene fecha de factura; solo el 28.6% tiene embarque confiable.
2. **Alineación con Finanzas**: *"El ingreso lo reconocemos por la fecha de la factura"* (correo de la contraparte).
3. **Consistencia interna**: al hacer la sensibilidad sobre el subconjunto comparable, la diferencia no altera las conclusiones.

### La respuesta a la pregunta de negocio

La tabla del punto 5 cruza crecimiento en volumen (S1→S2) con margen por litro bajo la Definición A, clasificando cada mercado en uno de cuatro cuadrantes. Esa clasificación **no cambia** si se usa la Definición B sobre los datos disponibles.